# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-Raza0/flyrank-ml-intership-hassan-raza/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Lane, two signals, and the rule

Lane 2 stays: **Refresh / Content Opportunity Scoring**. One row is one pseudonymized page. The slice is the Week 2 slice, `impressions_90d >= 100`. The proxy label is `trend_direction == "down"`. That label is used only to test the signals. It is not an input to the score.

The refresh flag in the session is staleness: a page that has not been updated. The second signal is volume, the signal behind a quick-win flag. Each bucket table below prints the decline rate and **n**.

**Staleness — MIXED.** On this slice the decline rate is 0.583 for pages updated within 30 days (n=13,735) and 0.623 for pages last updated 90 or more days ago (n=8,119). The session's 181+ day cut declines at 0.743, but **n=35**, so that cut cannot fill a queue. Of the 90-day group, **7,937 pages share the exact value 104 days**, almost all keyword articles. The direction matches the refresh flag. It is not a smooth age effect.

**Prior-window volume — OPPOSITE.** This check saved the rule from ranking by impressions. On the same slice, the excellent impression tier declines at 0.462 (n=1,078) and the low tier declines at 0.613 (n=3,254). More volume is not more decline. A score that multiplies by impressions would put the wrong end of the list first.

**The rule, in plain words.** Review a page for refresh when it has not been updated for at least 90 days and it had at least 100 impressions in the prior 30 days (days 31–60, before the latest window that builds the label). Rank those pages by days since update, not by impressions. Every flagged page gets one reason code, `stale_visible_prior_window`, and one action, `review_for_refresh`.


In [1]:
from pathlib import Path

import pandas as pd

here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")

# Proxy only. Never used as a score input.
df["declining"] = (df["trend_direction"] == "down").astype(int)
lane = df[df["impressions_90d"] >= 100].copy()
base_rate = lane["declining"].mean()
print(f"Lane rows: {len(lane):,}  |  declining base rate: {base_rate:.3f}")

fresh_order = ["0-30", "31-90", "91-180", "181+"]
fresh = (
    lane.groupby("freshness_tier")["declining"]
    .agg(decline_rate="mean", n="count")
    .reindex(fresh_order)
)
fresh["decline_rate"] = fresh["decline_rate"].round(3)
print("\nSignal 1 — staleness (refresh flag), by freshness tier")
print(fresh.to_string())

stamp = pd.Series(
    {
        "under_90_days": lane.loc[lane["days_since_last_update"] < 90, "declining"].mean(),
        "exactly_104_days": lane.loc[lane["days_since_last_update"] == 104, "declining"].mean(),
        "other_90_plus": lane.loc[
            (lane["days_since_last_update"] >= 90) & (lane["days_since_last_update"] != 104),
            "declining",
        ].mean(),
    }
)
stamp_n = pd.Series(
    {
        "under_90_days": int((lane["days_since_last_update"] < 90).sum()),
        "exactly_104_days": int((lane["days_since_last_update"] == 104).sum()),
        "other_90_plus": int(((lane["days_since_last_update"] >= 90) & (lane["days_since_last_update"] != 104)).sum()),
    }
)
stamp_table = pd.DataFrame({"decline_rate": stamp.round(3), "n": stamp_n})
print("\nThe 90-day group is mostly one repeated day count")
print(stamp_table.to_string())
print("Verdict: MIXED")

tier_order = ["low", "moderate", "good", "excellent"]
volume = (
    lane.groupby("impression_tier")["declining"]
    .agg(decline_rate="mean", n="count")
    .reindex(tier_order)
)
volume["decline_rate"] = volume["decline_rate"].round(3)
print("\nSignal 2 — volume (quick-win flag), by impression tier")
print(volume.to_string())
print("Verdict: OPPOSITE")


Lane rows: 22,006  |  declining base rate: 0.598

Signal 1 — staleness (refresh flag), by freshness tier
                decline_rate      n
freshness_tier                     
0-30                   0.583  13735
31-90                  0.592    152
91-180                 0.622   8084
181+                   0.743     35

The 90-day group is mostly one repeated day count
                  decline_rate      n
under_90_days            0.583  13887
exactly_104_days         0.623   7937
other_90_plus            0.643    182
Verdict: MIXED

Signal 2 — volume (quick-win flag), by impression tier
                 decline_rate      n
impression_tier                     
low                     0.613   3254
moderate                0.615  10469
good                    0.586   7205
excellent               0.462   1078
Verdict: OPPOSITE


## 2. The ranked queue

The score is `days_since_last_update` for pages that pass both gates, and 0 otherwise. The gates are `days_since_last_update >= 90` and `impressions_prev_30d >= 100`.

`impressions_prev_30d` is the window before the latest 30 days. The score does not use `trend_pct`, `trend_direction`, `impressions_last_30d`, or the declining label. Volume is a gate, not the sort key, because the volume check was OPPOSITE.

The CSV keeps only the flagged pages. Each of them has the same reason code and the same action.


In [2]:
import json

gate = (lane["days_since_last_update"] >= 90) & (lane["impressions_prev_30d"] >= 100)
queue = lane.loc[gate].copy()
queue["action_score"] = queue["days_since_last_update"]
queue["reason_code"] = "stale_visible_prior_window"
queue["recommended_action"] = "review_for_refresh"

# content_id only breaks ties. It is a pseudonym, not a reason to refresh.
queue = queue.sort_values(["action_score", "content_id"], ascending=[False, True])
queue.insert(0, "priority_rank", range(1, len(queue) + 1))

precision_at_50 = queue["declining"].head(50).mean()
flagged_rate = queue["declining"].mean()
print(f"Flagged pages: {len(queue):,}")
print(f"Decline rate inside the rule: {flagged_rate:.3f}")
print(f"Decline base rate on the slice: {base_rate:.3f}")
print(f"Precision@50 of this ranking: {precision_at_50:.3f}")
print("Reason codes:", queue["reason_code"].unique().tolist())
print("Actions:", queue["recommended_action"].unique().tolist())

export_columns = [
    "priority_rank",
    "content_id",
    "client_id",
    "action_score",
    "reason_code",
    "recommended_action",
    "days_since_last_update",
    "impressions_prev_30d",
    "avg_position",
    "ctr",
]
out_dir = root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"
queue[export_columns].to_csv(csv_path, index=False)
print("Wrote", csv_path.relative_to(root), "rows", len(queue))

score_inputs = {"days_since_last_update", "impressions_prev_30d"}
forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "declining"}
print("Score inputs:", sorted(score_inputs))
print("Forbidden inputs used:", sorted(score_inputs & forbidden))

metrics = {
    "lane": "Lane 2 Refresh / Content Opportunity Scoring",
    "slice": "impressions_90d >= 100",
    "slice_rows": int(len(lane)),
    "base_rate_declining": round(float(base_rate), 4),
    "flagged_rows": int(len(queue)),
    "flagged_decline_rate": round(float(flagged_rate), 4),
    "precision_at_50": round(float(precision_at_50), 4),
    "reason_code": "stale_visible_prior_window",
    "action": "review_for_refresh",
    "signals": [
        {"name": "staleness_days_since_last_update", "flag": "refresh", "verdict": "MIXED"},
        {"name": "impression_tier_volume", "flag": "quick-win", "verdict": "OPPOSITE"},
    ],
}
(out_dir / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(queue[export_columns].head(10).to_string(index=False))


Flagged pages: 7,025
Decline rate inside the rule: 0.629
Decline base rate on the slice: 0.598
Precision@50 of this ranking: 0.900
Reason codes: ['stale_visible_prior_window']
Actions: ['review_for_refresh']
Wrote work\outputs\baseline_action_score.csv rows 7025
Score inputs: ['days_since_last_update', 'impressions_prev_30d']
Forbidden inputs used: []
 priority_rank           content_id         client_id  action_score                reason_code recommended_action  days_since_last_update  impressions_prev_30d  avg_position  ctr
             1 content_02b0d6e30129 client_19581e27de           313 stale_visible_prior_window review_for_refresh                     313                   159           6.9 0.00
             2 content_72496874f806 client_4ec9599fc2           301 stale_visible_prior_window review_for_refresh                     301                   125           5.8 0.24
             3 content_7f116ae1f6f5 client_9400f1b21c           301 stale_visible_prior_window review_for_ref

## 3. Top-10 review

For each of the first ten rows: the action, why the rule put it there, and what would make that call wrong. CTR is a ×100 percentage, so 0.24 means 0.24%. Position 0 would mean no position data. None of these ten use the declining label as a reason to rank.


In [3]:
top10 = queue.head(10)

def what_would_make_it_wrong(row):
    notes = []
    day = int(row.days_since_last_update)
    shared = int((lane["days_since_last_update"] == day).sum())
    if shared >= 5:
        notes.append(
            f"{shared} pages on this slice share the exact value {day} days, so the count may be a batch stamp"
        )
    if row.avg_position >= 20:
        notes.append(
            f"average position is {row.avg_position}, deep enough that a refresh may not be what an editor can move"
        )
    if row.avg_position == 0:
        notes.append("average position is 0, which means no position data")
    if row.ctr == 0:
        notes.append("CTR is 0.00 percent, which may be a measurement gap rather than a stale article")
    if row.ctr >= 2:
        notes.append(
            f"CTR is already {row.ctr} percent, so a refresh could disturb a page that is being clicked"
        )
    if row.priority_rank > 1 and row.action_score == top10.iloc[0].action_score:
        notes.append("the tie is broken by content_id, which is a pseudonym and not a reason to go first")
    if not notes:
        notes.append("the prior-window impressions may belong to queries this refresh will not touch")
    return "; ".join(notes)

print("priority | action | why it is here | what would make it wrong")
for row in top10.itertuples(index=False):
    why = (
        f"{int(row.days_since_last_update)} days since update and "
        f"{int(row.impressions_prev_30d)} impressions in the prior 30 days "
        f"(position {row.avg_position}, CTR {row.ctr} percent)"
    )
    print(
        f"{int(row.priority_rank)}. {row.recommended_action} | {row.content_id} | "
        f"{why} | {what_would_make_it_wrong(row)}"
    )


priority | action | why it is here | what would make it wrong
1. review_for_refresh | content_02b0d6e30129 | 313 days since update and 159 impressions in the prior 30 days (position 6.9, CTR 0.0 percent) | CTR is 0.00 percent, which may be a measurement gap rather than a stale article
2. review_for_refresh | content_72496874f806 | 301 days since update and 125 impressions in the prior 30 days (position 5.8, CTR 0.24 percent) | the prior-window impressions may belong to queries this refresh will not touch
3. review_for_refresh | content_7f116ae1f6f5 | 301 days since update and 339 impressions in the prior 30 days (position 9.0, CTR 0.42 percent) | the prior-window impressions may belong to queries this refresh will not touch
4. review_for_refresh | content_4f241bad48a3 | 236 days since update and 187 impressions in the prior 30 days (position 19.1, CTR 0.0 percent) | CTR is 0.00 percent, which may be a measurement gap rather than a stale article
5. review_for_refresh | content_ba00ffc63

## 4. Weak picks and the leakage check

The top of this list is a thin tail, not the whole rule. Precision@50 is about 0.90 because those 50 pages are the oldest ones. The rule flags 7,025 pages, and their decline rate is about 0.629 against a slice base rate of about 0.598. Beating the top-50 number is not the same as beating a coin-flip on the full queue.

Weak picks inside the top ten are the pages with a deep average position, a CTR of 0.00 percent, or a day-count shared by several pages. The tie-break is `content_id`. That can change who sits in the top ten without changing anything an editor would call a reason.

The score does not read `trend_pct`, `trend_direction`, `impressions_last_30d`, or the declining label. The lane filter still uses `impressions_90d`, which overlaps the 90-day window the label is built from. That filter matches Week 2. It is not part of the score.


In [4]:
used_in_score = ["days_since_last_update", "impressions_prev_30d"]
blocked = ["trend_pct", "trend_direction", "impressions_last_30d", "declining"]
print("Used in the score:", used_in_score)
print("Blocked columns present in the score:", [name for name in blocked if name in used_in_score])
deep = top10[top10["avg_position"] >= 20]
zero_ctr = top10[top10["ctr"] == 0]
print(f"Top 10 with average position >= 20: {len(deep)}")
print(f"Top 10 with CTR of 0.00 percent: {len(zero_ctr)}")
print(
    "Full flagged decline rate "
    f"{queue['declining'].mean():.3f} versus top-50 "
    f"{queue['declining'].head(50).mean():.3f}"
)


Used in the score: ['days_since_last_update', 'impressions_prev_30d']
Blocked columns present in the score: []
Top 10 with average position >= 20: 4
Top 10 with CTR of 0.00 percent: 3
Full flagged decline rate 0.629 versus top-50 0.900


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
